In [0]:
from pyspark.sql.functions import col, current_timestamp

VOL = "/Volumes/meridian_dev/bronze/raw_files"

# metadata-driven: one list controls all the loads
tables = ["encounters", "conditions", "medications", "procedures", "payers"]

for t in tables:
    df = (spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(f"{VOL}/{t}.csv")
        .withColumn("source_file", col("_metadata.file_name"))
        .withColumn("ingested_at", current_timestamp()))

    df.write.format("delta").mode("overwrite").saveAsTable(f"meridian_dev.bronze.{t}")
    print(f"Loaded bronze.{t}: {df.count()} rows")

print("\nAll core bronze tables loaded.")

In [0]:
%sql
SHOW TABLES IN meridian_dev.bronze;